# CRN_Light (Merged) — MVDR Post-Enhancement Training

This notebook is a **single-file merge** of your:
- `crn_light.py` (model)
- `stft_utils.py` (STFT/iSTFT)
- `dataset.py` (MVDRDataset)
- `losses.py` (losses)
- `train_crn.py` (training loop)

Changes requested:
1) **Checkpointing + auto-resume**: saves every `SAVE_EVERY_EPOCHS`, and if you rerun, it loads the latest checkpoint and resumes from the next epoch.
2) **Progress bars**: updates after every batch and **persists** when finished (does not vanish).

On Kaggle, set `MVDR_DIR` / `CLEAN_DIR` to your dataset paths (usually under `/kaggle/input/...`).

In [ ]:
# (Optional) Kaggle usually has these already. Uncomment if needed.
# !pip -q install soundfile tqdm

import os
import re
import glob
import math
from dataclasses import dataclass

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset

from tqdm.auto import tqdm

try:
    import torchaudio
    _HAS_TORCHAUDIO = True
except ModuleNotFoundError:
    torchaudio = None
    _HAS_TORCHAUDIO = False

try:
    import soundfile as sf
    _HAS_SOUNDFILE = True
except ModuleNotFoundError:
    sf = None
    _HAS_SOUNDFILE = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print('Device:', DEVICE)
print('Torch:', torch.__version__)

## Model (from `crn_light.py`)

In [ ]:
class DepthwiseSeparableConv2d(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size=(3, 3),
        stride=(1, 1),
        padding=(1, 1),
        bias: bool = True,
    ):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_channels,
            in_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            groups=in_channels,
            bias=bias,
        )
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=bias)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x


class InvertedResidual2d(nn.Module):
    def __init__(
        self,
        channels: int,
        expansion: int = 4,
        kernel_size=(3, 3),
        causal: bool = False,
        bias: bool = True,
    ):
        super().__init__()
        hidden = channels * expansion
        self.causal = causal
        self.kernel_size = kernel_size

        self.expand = nn.Conv2d(channels, hidden, kernel_size=1, bias=bias)
        self.dw = nn.Conv2d(
            hidden,
            hidden,
            kernel_size=kernel_size,
            stride=1,
            padding=0 if causal else (kernel_size[0] // 2, kernel_size[1] // 2),
            groups=hidden,
            bias=bias,
        )
        self.project = nn.Conv2d(hidden, channels, kernel_size=1, bias=bias)

    def forward(self, x):
        residual = x
        x = F.relu(self.expand(x))

        if self.causal:
            # Causal along time (last dim). Keep freq padding symmetric.
            pad_f = self.kernel_size[0] // 2
            pad_t = self.kernel_size[1] - 1
            x = F.pad(x, (pad_t, 0, pad_f, pad_f))

        x = F.relu(self.dw(x))
        x = self.project(x)
        return x + residual


class CRN_Light(nn.Module):
    """
    Tiny CRN-like U-Net for MVDR post-enhancement
    ~0.15M parameters (target for on-device)
    Input:  [B, 2, F, T] where channels = [real, imag]
    Output: [B, 2, F, T] complex mask [m_r, m_i]
    """

    def __init__(
        self,
        freq_bins: int = 257,
        in_channels: int = 2,
        out_channels: int = 2,
        base_channels: int = 16,
        bottleneck_channels: int = 48,
        bottleneck_blocks: int = 6,
        bottleneck_expansion: int = 4,
        causal: bool = False,
    ):
        super().__init__()

        c1 = base_channels
        c2 = int(round(base_channels * 1.5))
        c3 = int(round(base_channels * 2.0))
        c4 = bottleneck_channels

        self.in_channels = in_channels
        self.out_channels = out_channels

        self.enc1 = DepthwiseSeparableConv2d(in_channels, c1, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        self.enc2 = DepthwiseSeparableConv2d(c1, c2, kernel_size=(3, 3), stride=(2, 1), padding=(1, 1))
        self.enc3 = DepthwiseSeparableConv2d(c2, c3, kernel_size=(3, 3), stride=(2, 1), padding=(1, 1))
        self.enc4 = DepthwiseSeparableConv2d(c3, c4, kernel_size=(3, 3), stride=(2, 1), padding=(1, 1))

        # Three strided layers reduce F as ceil(F/8).
        self.freq_reduced = (freq_bins + 7) // 8

        self.bottleneck = nn.Sequential(
            *[
                InvertedResidual2d(
                    channels=c4,
                    expansion=bottleneck_expansion,
                    kernel_size=(3, 3),
                    causal=causal,
                )
                for _ in range(bottleneck_blocks)
            ]
        )

        self.up4 = nn.Upsample(scale_factor=(2, 1), mode="nearest")
        self.dec4 = DepthwiseSeparableConv2d(c4 + c3, c3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))

        self.up3 = nn.Upsample(scale_factor=(2, 1), mode="nearest")
        self.dec3 = DepthwiseSeparableConv2d(c3 + c2, c2, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))

        self.up2 = nn.Upsample(scale_factor=(2, 1), mode="nearest")
        self.dec2 = DepthwiseSeparableConv2d(c2 + c1, c1, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))

        self.out = nn.Conv2d(c1, out_channels, kernel_size=1)

        # Start close to identity for complex masking: m_r≈1, m_i≈0.
        nn.init.zeros_(self.out.weight)
        if self.out.bias is not None:
            nn.init.zeros_(self.out.bias)

    def forward(self, x):
        # x: [B,2,F,T]

        def _align_to(ref, y):
            """Crop/pad y on (freq,time) dims to match ref."""
            ref_f, ref_t = ref.shape[2], ref.shape[3]
            y_f, y_t = y.shape[2], y.shape[3]

            if y_f > ref_f:
                y = y[:, :, :ref_f, :]
            elif y_f < ref_f:
                y = F.pad(y, (0, 0, 0, ref_f - y_f))

            if y_t > ref_t:
                y = y[:, :, :, :ref_t]
            elif y_t < ref_t:
                y = F.pad(y, (0, ref_t - y_t, 0, 0))

            return y

        e1 = F.relu(self.enc1(x))
        e2 = F.relu(self.enc2(e1))
        e3 = F.relu(self.enc3(e2))
        e4 = F.relu(self.enc4(e3))

        b = self.bottleneck(e4)

        d4 = self.up4(b)
        d4 = _align_to(e3, d4)
        d4 = F.relu(self.dec4(torch.cat([d4, e3], dim=1)))

        d3 = self.up3(d4)
        d3 = _align_to(e2, d3)
        d3 = F.relu(self.dec3(torch.cat([d3, e2], dim=1)))

        d2 = self.up2(d3)
        d2 = _align_to(e1, d2)
        d2 = F.relu(self.dec2(torch.cat([d2, e1], dim=1)))

        mask = torch.tanh(self.out(d2))

        if mask.shape[1] == 2:
            m_r = 1.0 + 0.5 * mask[:, 0:1]
            m_i = 0.5 * mask[:, 1:2]
            return torch.cat([m_r, m_i], dim=1)

        return mask

## STFT utils (from `stft_utils.py`)

In [ ]:
def stft_mag(wav, device=None):
    if device is None:
        device = wav.device
    window = torch.sqrt(torch.hann_window(256, periodic=True)).to(device)

    spec = torch.stft(
        wav,
        n_fft=512,
        hop_length=128,
        win_length=256,
        window=window,
        center=True,
        return_complex=True,
    )
    return torch.abs(spec), spec


def istft(spec, device=None, length=None):
    if device is None:
        device = spec.device
    window = torch.sqrt(torch.hann_window(256, periodic=True)).to(device)

    return torch.istft(
        spec,
        n_fft=512,
        hop_length=128,
        win_length=256,
        window=window,
        center=True,
        length=length,
    )

## Dataset (from `dataset.py`)

In [ ]:
def _load_wav(path: str, target_sr: int) -> torch.Tensor:
    # Prefer soundfile when available: lightweight + avoids optional torchaudio decoder deps.
    if _HAS_SOUNDFILE:
        data, sr = sf.read(path, dtype="float32", always_2d=True)  # [N, C]
        if sr != target_sr:
            if _HAS_TORCHAUDIO:
                wav = torch.from_numpy(np.asarray(data).T)  # [C, N]
                wav = torchaudio.functional.resample(wav, sr, target_sr)
                sr = target_sr
            else:
                raise ValueError(
                    f"Sample rate mismatch for {path}: got {sr}, expected {target_sr}. "
                    "Install torchaudio to enable resampling."
                )
        else:
            wav = torch.from_numpy(np.asarray(data).T)
    elif _HAS_TORCHAUDIO:
        wav, sr = torchaudio.load(path)
        if sr != target_sr:
            wav = torchaudio.functional.resample(wav, sr, target_sr)
        wav = wav.to(torch.float32)
    else:
        raise ModuleNotFoundError("Missing audio backend. Install soundfile or torchaudio.")

    if wav.dim() != 2:
        raise RuntimeError(f"Expected waveform shape [C,N], got {tuple(wav.shape)} for {path}")

    # Mix to mono if multi-channel
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0)
    else:
        wav = wav.squeeze(0)

    return wav


class MVDRDataset(Dataset):
    def __init__(self, mvdr_dir, clean_dir, sample_rate=16000):
        self.mvdr_dir = mvdr_dir
        self.clean_dir = clean_dir
        self.sr = sample_rate

        self.mvdr_files = sorted([f for f in os.listdir(mvdr_dir) if f.endswith(".wav")])

        # Index clean references by stem (supports .wav/.flac)
        self.clean_index = {}
        for f in os.listdir(clean_dir):
            lower = f.lower()
            if not (lower.endswith(".wav") or lower.endswith(".flac")):
                continue
            stem, _ = os.path.splitext(f)
            self.clean_index[stem] = os.path.join(clean_dir, f)

    def _get_clean_stem(self, mvdr_name):
        # 1_part13_A_female_only.wav → 1_part13
        return "_".join(mvdr_name.split("_")[:2])

    def __len__(self):
        return len(self.mvdr_files)

    def __getitem__(self, idx):
        mvdr_name = self.mvdr_files[idx]
        clean_stem = self._get_clean_stem(mvdr_name)

        mvdr_path = os.path.join(self.mvdr_dir, mvdr_name)
        clean_path = self.clean_index.get(clean_stem)

        if not os.path.isfile(mvdr_path):
            raise FileNotFoundError(f"Missing MVDR wav: {mvdr_path}")
        if clean_path is None or not os.path.isfile(clean_path):
            raise FileNotFoundError(
                f"Missing clean reference for '{mvdr_name}'. Expected '{clean_stem}.wav' or '{clean_stem}.flac' in {self.clean_dir}"
            )

        mvdr_wav = _load_wav(mvdr_path, self.sr)
        clean_wav = _load_wav(clean_path, self.sr)

        # Ensure paired signals have identical length
        min_len = min(mvdr_wav.shape[-1], clean_wav.shape[-1])
        mvdr_wav = mvdr_wav[:min_len]
        clean_wav = clean_wav[:min_len]

        # Normalize both using MVDR scale (preserves relative loudness).
        scale = mvdr_wav.std() + 1e-8
        mvdr_wav = mvdr_wav / scale
        clean_wav = clean_wav / scale

        return mvdr_wav, clean_wav

## Losses (from `losses.py`)

In [ ]:
def log_mag_loss(est, ref):
    return F.l1_loss(torch.log(est + 1e-8), torch.log(ref + 1e-8))


def si_sdr_loss(est, ref, eps=1e-8):
    ref = ref - ref.mean(dim=-1, keepdim=True)
    est = est - est.mean(dim=-1, keepdim=True)

    proj = (torch.sum(est * ref, dim=-1, keepdim=True) * ref) / (torch.sum(ref ** 2, dim=-1, keepdim=True) + eps)
    noise = est - proj
    ratio = torch.sum(proj ** 2, dim=-1) / (torch.sum(noise ** 2, dim=-1) + eps)

    return (-10.0 * torch.log10(ratio + eps)).mean()


def _stft(x, n_fft, hop_length, win_length, window):
    return torch.stft(
        x,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window=window,
        center=True,
        return_complex=True,
    )


def mrstft_loss(
    est_wav: torch.Tensor,
    ref_wav: torch.Tensor,
    fft_sizes=(512, 1024, 2048),
    hop_ratios=(0.25, 0.25, 0.25),
    win_lengths=None,
    eps: float = 1e-8,
):
    """Multi-resolution STFT loss: spectral convergence + log-mag L1.

    Shapes: est_wav/ref_wav: [B, T]
    """
    if win_lengths is None:
        win_lengths = fft_sizes
    device = est_wav.device

    sc_total = 0.0
    mag_total = 0.0
    n = 0

    for n_fft, hop_ratio, win_length in zip(fft_sizes, hop_ratios, win_lengths):
        hop = int(round(n_fft * hop_ratio))
        window = torch.hann_window(win_length, periodic=True, device=device)

        est = _stft(est_wav, n_fft=n_fft, hop_length=hop, win_length=win_length, window=window)
        ref = _stft(ref_wav, n_fft=n_fft, hop_length=hop, win_length=win_length, window=window)

        est_mag = torch.abs(est)
        ref_mag = torch.abs(ref)

        sc = torch.norm(ref_mag - est_mag, p="fro") / (torch.norm(ref_mag, p="fro") + eps)
        mag = F.l1_loss(torch.log(est_mag + eps), torch.log(ref_mag + eps))

        sc_total = sc_total + sc
        mag_total = mag_total + mag
        n += 1

    sc_total = sc_total / max(1, n)
    mag_total = mag_total / max(1, n)
    return sc_total + mag_total

## Training + Checkpointing (merged from `train_crn.py`)

Key additions:
- Saves a full checkpoint dict (model + optimizer + epoch) every `SAVE_EVERY_EPOCHS`.
- On rerun, auto-loads the **latest epoch checkpoint** and resumes.
- Uses `tqdm.auto` with `leave=True`, `miniters=1`, `mininterval=0` so batch progress updates and persists.

In [ ]:
# -------------------- CONFIG (edit values here) --------------------
# For Kaggle, set these to your dataset folder paths.
MVDR_DIR = 'mvdr_outputs'
CLEAN_DIR = 'true_labels'

SAMPLE_RATE = 16000

BATCH_SIZE = 8
EPOCHS = 50
LR = 2e-4

SISDR_WARMUP_EPOCHS = 0
SI_SDR_WEIGHT_MAX = 1.0
MRSTFT_WEIGHT = 0.2
LOGMAG_WEIGHT = 0.1
IMPROVEMENT_HINGE_WEIGHT = 0.5

TRAIN_RATIO = 0.85
SEED = 1337

NUM_WORKERS = 4 if DEVICE == 'cuda' else 0

SEGMENT_SECONDS = 2.0
SEGMENT_SAMPLES = int(SAMPLE_RATE * SEGMENT_SECONDS)
SEGMENT_CANDIDATES = 8

VAL_FULL_UTTERANCE = True

MAX_TRAIN_BATCHES_PER_EPOCH = None
MAX_VAL_BATCHES_PER_EPOCH = None

SAVE_EVERY_EPOCHS = 1
CHECKPOINT_PREFIX = 'crn'
CHECKPOINT_DIR = 'checkpoints'
AUTO_RESUME = True
# -------------------------------------------------------------------

def _seed_everything(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _pad_collate(batch):
    mvdr_list, clean_list = zip(*batch)
    mvdr_out = []
    clean_out = []

    for mvdr_wav, clean_wav in zip(mvdr_list, clean_list):
        if SEGMENT_SAMPLES is None:
            mvdr_out.append(mvdr_wav)
            clean_out.append(clean_wav)
            continue

        length = mvdr_wav.shape[-1]
        if length >= SEGMENT_SAMPLES:
            max_start = length - SEGMENT_SAMPLES
            if max_start == 0:
                start = 0
            else:
                candidates = torch.linspace(0, max_start, steps=min(SEGMENT_CANDIDATES, max_start + 1)).long()
                best_start = 0
                best_energy = None
                for s in candidates.tolist():
                    seg = clean_wav[s : s + SEGMENT_SAMPLES]
                    energy = torch.mean(seg * seg)
                    if best_energy is None or energy > best_energy:
                        best_energy = energy
                        best_start = s
                start = best_start

            mvdr_wav = mvdr_wav[start : start + SEGMENT_SAMPLES]
            clean_wav = clean_wav[start : start + SEGMENT_SAMPLES]
        else:
            pad = SEGMENT_SAMPLES - length
            mvdr_wav = torch.nn.functional.pad(mvdr_wav, (0, pad))
            clean_wav = torch.nn.functional.pad(clean_wav, (0, pad))

        mvdr_out.append(mvdr_wav)
        clean_out.append(clean_wav)

    return torch.stack(mvdr_out, dim=0), torch.stack(clean_out, dim=0)


def _train_collate(batch):
    mvdr_list, clean_list = zip(*batch)
    mvdr_out = []
    clean_out = []

    for mvdr_wav, clean_wav in zip(mvdr_list, clean_list):
        if SEGMENT_SAMPLES is None:
            mvdr_out.append(mvdr_wav)
            clean_out.append(clean_wav)
            continue

        length = mvdr_wav.shape[-1]
        if length >= SEGMENT_SAMPLES:
            max_start = length - SEGMENT_SAMPLES
            if max_start == 0:
                start = 0
            else:
                best_start = 0
                best_energy = None
                for _ in range(max(1, SEGMENT_CANDIDATES)):
                    s = int(torch.randint(low=0, high=max_start + 1, size=(1,)).item())
                    seg = clean_wav[s : s + SEGMENT_SAMPLES]
                    energy = torch.mean(seg * seg)
                    if best_energy is None or energy > best_energy:
                        best_energy = energy
                        best_start = s
                start = best_start

            mvdr_wav = mvdr_wav[start : start + SEGMENT_SAMPLES]
            clean_wav = clean_wav[start : start + SEGMENT_SAMPLES]
        else:
            pad = SEGMENT_SAMPLES - length
            mvdr_wav = torch.nn.functional.pad(mvdr_wav, (0, pad))
            clean_wav = torch.nn.functional.pad(clean_wav, (0, pad))

        mvdr_out.append(mvdr_wav)
        clean_out.append(clean_wav)

    return torch.stack(mvdr_out, dim=0), torch.stack(clean_out, dim=0)


def _validate_paths():
    if not os.path.isdir(MVDR_DIR):
        raise FileNotFoundError(f'MVDR_DIR not found: {MVDR_DIR}')
    if not os.path.isdir(CLEAN_DIR):
        raise FileNotFoundError(f'CLEAN_DIR not found: {CLEAN_DIR}')

    mvdr_wavs = glob.glob(os.path.join(MVDR_DIR, '*.wav'))
    clean_wavs = glob.glob(os.path.join(CLEAN_DIR, '*.wav')) + glob.glob(os.path.join(CLEAN_DIR, '*.flac'))
    if len(mvdr_wavs) == 0:
        raise RuntimeError(f'No .wav files found in {MVDR_DIR}')
    if len(clean_wavs) == 0:
        raise RuntimeError(f'No .wav/.flac files found in {CLEAN_DIR}')


def _make_loaders():
    dataset = MVDRDataset(MVDR_DIR, CLEAN_DIR, sample_rate=SAMPLE_RATE)
    if len(dataset) < 2:
        raise RuntimeError(f'Need at least 2 samples to split train/val, got {len(dataset)}')

    train_len = int(round(len(dataset) * TRAIN_RATIO))
    train_len = max(1, min(train_len, len(dataset) - 1))
    val_len = len(dataset) - train_len

    generator = torch.Generator().manual_seed(SEED)
    train_set, val_set = random_split(dataset, [train_len, val_len], generator=generator)

    train_loader = DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == 'cuda'),
        collate_fn=_train_collate,
    )

    if VAL_FULL_UTTERANCE:
        val_loader = DataLoader(
            val_set,
            batch_size=1,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=(DEVICE == 'cuda'),
        )
    else:
        val_loader = DataLoader(
            val_set,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=(DEVICE == 'cuda'),
            collate_fn=_pad_collate,
        )

    return train_loader, val_loader


def _sisdr_weight_for_epoch(epoch_idx: int) -> float:
    if SISDR_WARMUP_EPOCHS <= 0:
        return SI_SDR_WEIGHT_MAX
    if epoch_idx <= SISDR_WARMUP_EPOCHS:
        return 0.0
    ramp = min(1.0, (epoch_idx - SISDR_WARMUP_EPOCHS) / 5.0)
    return SI_SDR_WEIGHT_MAX * ramp


def _forward_losses(model, mvdr_wav, clean_wav, epoch_idx: int):
    _, mvdr_complex = stft_mag(mvdr_wav, DEVICE)
    clean_mag, _ = stft_mag(clean_wav, DEVICE)

    x = torch.stack([mvdr_complex.real, mvdr_complex.imag], dim=1)  # [B,2,F,T]

    m = model(x)
    m_r = m[:, 0, :, :]
    m_i = m[:, 1, :, :]

    x_r = mvdr_complex.real
    x_i = mvdr_complex.imag

    y_r = m_r * x_r - m_i * x_i
    y_i = m_r * x_i + m_i * x_r
    est_complex = torch.complex(y_r, y_i)

    est_wav = istft(est_complex, DEVICE, length=clean_wav.shape[-1])

    est_mag = torch.abs(est_complex).unsqueeze(1)
    clean_mag = clean_mag.unsqueeze(1)

    loss_logmag = log_mag_loss(est_mag, clean_mag)
    loss_sisdr = si_sdr_loss(est_wav, clean_wav)
    loss_mrstft = mrstft_loss(est_wav, clean_wav)

    baseline_sisdr_loss = si_sdr_loss(mvdr_wav, clean_wav)
    improvement_hinge = torch.relu(loss_sisdr - baseline_sisdr_loss)

    si_sdr_weight = _sisdr_weight_for_epoch(epoch_idx)

    total_loss = (
        si_sdr_weight * loss_sisdr
        + MRSTFT_WEIGHT * loss_mrstft
        + LOGMAG_WEIGHT * loss_logmag
        + IMPROVEMENT_HINGE_WEIGHT * improvement_hinge
    )

    return total_loss, loss_logmag, loss_sisdr, baseline_sisdr_loss


def _make_pbar(iterable, desc: str):
    # Updates after every batch and persists after finishing.
    return tqdm(
        iterable,
        desc=desc,
        leave=True,
        miniters=1,
        mininterval=0.0,
        smoothing=0.0,
        dynamic_ncols=True,
    )


def train_one_epoch(model, optimizer, train_loader, epoch_idx):
    model.train()

    total = 0.0
    total_mag = 0.0
    total_sisdr = 0.0
    total_base_sisdr = 0.0
    n = 0

    pbar = _make_pbar(train_loader, desc=f'Train {epoch_idx:03d}')
    for batch_idx, (mvdr_wav, clean_wav) in enumerate(pbar, start=1):
        mvdr_wav = mvdr_wav.to(DEVICE)
        clean_wav = clean_wav.to(DEVICE)

        loss, loss_mag, loss_sisdr, base_sisdr = _forward_losses(model, mvdr_wav, clean_wav, epoch_idx)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        total += float(loss.item())
        total_mag += float(loss_mag.item())
        total_sisdr += float(loss_sisdr.item())
        total_base_sisdr += float(base_sisdr.item())
        n += 1

        est_db = -(total_sisdr / n)
        base_db = -(total_base_sisdr / n)
        imp_db = est_db - base_db

        pbar.set_postfix(
            loss=f'{total/n:.3f}',
            mag=f'{total_mag/n:.3f}',
            sisdr_db=f'{est_db:.2f}',
            imp_db=f'{imp_db:.2f}',
        )

        if MAX_TRAIN_BATCHES_PER_EPOCH is not None and batch_idx >= MAX_TRAIN_BATCHES_PER_EPOCH:
            break

    denom = max(1, n)
    return total / denom, total_mag / denom, total_sisdr / denom, total_base_sisdr / denom


@torch.no_grad()
def validate(model, val_loader, epoch_idx):
    model.eval()

    total = 0.0
    total_mag = 0.0
    total_sisdr = 0.0
    total_base_sisdr = 0.0
    n = 0

    pbar = _make_pbar(val_loader, desc=f'Val   {epoch_idx:03d}')
    for batch_idx, (mvdr_wav, clean_wav) in enumerate(pbar, start=1):
        mvdr_wav = mvdr_wav.to(DEVICE)
        clean_wav = clean_wav.to(DEVICE)

        loss, loss_mag, loss_sisdr, base_sisdr = _forward_losses(model, mvdr_wav, clean_wav, epoch_idx)

        total += float(loss.item())
        total_mag += float(loss_mag.item())
        total_sisdr += float(loss_sisdr.item())
        total_base_sisdr += float(base_sisdr.item())
        n += 1

        est_db = -(total_sisdr / n)
        base_db = -(total_base_sisdr / n)
        imp_db = est_db - base_db

        pbar.set_postfix(
            loss=f'{total/n:.3f}',
            mag=f'{total_mag/n:.3f}',
            sisdr_db=f'{est_db:.2f}',
            imp_db=f'{imp_db:.2f}',
        )

        if MAX_VAL_BATCHES_PER_EPOCH is not None and batch_idx >= MAX_VAL_BATCHES_PER_EPOCH:
            break

    denom = max(1, n)
    return total / denom, total_mag / denom, total_sisdr / denom, total_base_sisdr / denom


def _checkpoint_paths(epoch: int):
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    epoch_path = os.path.join(CHECKPOINT_DIR, f'{CHECKPOINT_PREFIX}_epoch_{epoch:03d}.pt')
    latest_path = os.path.join(CHECKPOINT_DIR, f'{CHECKPOINT_PREFIX}_latest.pt')
    return epoch_path, latest_path


def save_checkpoint(epoch: int, model: nn.Module, optimizer: optim.Optimizer):
    epoch_path, latest_path = _checkpoint_paths(epoch)

    payload = {
        'epoch': int(epoch),
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'config': {
            'sample_rate': SAMPLE_RATE,
            'batch_size': BATCH_SIZE,
            'lr': LR,
        },
    }

    torch.save(payload, epoch_path)
    torch.save(payload, latest_path)
    return epoch_path


def _parse_epoch_from_filename(path: str):
    m = re.search(r'_epoch_(\d+)\.pt$', os.path.basename(path))
    if not m:
        return None
    return int(m.group(1))


def find_latest_checkpoint():
    # Prefer explicit latest if present, else scan epoch files.
    latest_path = os.path.join(CHECKPOINT_DIR, f'{CHECKPOINT_PREFIX}_latest.pt')
    if os.path.isfile(latest_path):
        return latest_path

    pattern = os.path.join(CHECKPOINT_DIR, f'{CHECKPOINT_PREFIX}_epoch_*.pt')
    candidates = glob.glob(pattern)
    if not candidates:
        return None

    best = None
    best_epoch = -1
    for p in candidates:
        ep = _parse_epoch_from_filename(p)
        if ep is None:
            continue
        if ep > best_epoch:
            best_epoch = ep
            best = p

    return best


def maybe_resume(model: nn.Module, optimizer: optim.Optimizer):
    if not AUTO_RESUME:
        return 1

    ckpt_path = find_latest_checkpoint()
    if ckpt_path is None:
        return 1

    ckpt = torch.load(ckpt_path, map_location=DEVICE)

    # Backward compatible: if someone saved raw state_dict only.
    if isinstance(ckpt, dict) and 'model_state' in ckpt:
        model.load_state_dict(ckpt['model_state'])
        if 'optimizer_state' in ckpt:
            try:
                optimizer.load_state_dict(ckpt['optimizer_state'])
            except Exception as e:
                print('Warning: could not load optimizer state:', e)
        last_epoch = int(ckpt.get('epoch', 0))
    else:
        model.load_state_dict(ckpt)
        last_epoch = _parse_epoch_from_filename(ckpt_path) or 0

    print(f'Resuming from checkpoint: {ckpt_path} (epoch {last_epoch})')
    return last_epoch + 1


def main():
    _seed_everything(SEED)
    _validate_paths()

    train_loader, val_loader = _make_loaders()

    model = CRN_Light().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    start_epoch = maybe_resume(model, optimizer)

    print(f'Device: {DEVICE}')
    print(f'Train/Val sizes: {len(train_loader.dataset)}/{len(val_loader.dataset)}')
    print(f'Val full utterance: {VAL_FULL_UTTERANCE}')
    print(f'Total parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')
    print(f'Start epoch: {start_epoch} / Target epochs: {EPOCHS}')

    if start_epoch > EPOCHS:
        print('Nothing to do: latest checkpoint is already >= EPOCHS.')
        return

    for epoch in range(start_epoch, EPOCHS + 1):
        tr_loss, tr_mag, tr_sisdr, tr_base = train_one_epoch(model, optimizer, train_loader, epoch)
        va_loss, va_mag, va_sisdr, va_base = validate(model, val_loader, epoch)

        tr_sisdr_db = -tr_sisdr
        va_sisdr_db = -va_sisdr
        tr_base_db = -tr_base
        va_base_db = -va_base

        print(
            f'Epoch {epoch:03d} | '
            f'train: loss={tr_loss:.4f}, base={tr_base_db:.2f}dB, enh={tr_sisdr_db:.2f}dB, imp={tr_sisdr_db - tr_base_db:.2f}dB | '
            f'val: loss={va_loss:.4f}, base={va_base_db:.2f}dB, enh={va_sisdr_db:.2f}dB, imp={va_sisdr_db - va_base_db:.2f}dB'
        )

        if SAVE_EVERY_EPOCHS and (epoch % SAVE_EVERY_EPOCHS == 0):
            path = save_checkpoint(epoch, model, optimizer)
            print(f'Saved checkpoint: {path}')

    print('Done.')

In [ ]:
# Run training
main()